# 022 — Sistemas expertos y motores de reglas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

Un **sistema experto** separa conocimiento y mecanismo: base de reglas
`SI condiciones ENTONCES conclusión`, memoria de trabajo (hechos), motor de
inferencia y componente de explicación.

- **Encadenamiento hacia adelante** (dirigido por datos): ciclo
  *match-resolve-act* hasta el punto fijo; completo para cláusulas de Horn.
- **Encadenamiento hacia atrás** (dirigido por objetivos): parte de la
  hipótesis y genera subobjetivos (Prolog, MYCIN).
- **Rete** (Forgy, 1982): compila las condiciones en una red con memorias
  alfa/beta y propaga solo los **cambios** de la memoria de trabajo — cambia
  memoria por velocidad; base de CLIPS y Drools.
- **Factores de certeza (MYCIN)**: `CF ∈ [-1, 1]`; encadenado
  `CF = CF_regla × CF_premisa`; evidencia paralela
  `CF = CF1 + CF2·(1−CF1)`. No son probabilidades: asumen independencia.


## 🧮 La base de reglas del laboratorio

```text
hechos iniciales: {tiene_datos, tiene_objetivo}
R1: tiene_datos ∧ tiene_objetivo → puede_experimentar
R2: puede_experimentar          → requiere_baseline
R3: requiere_baseline           → requiere_evaluacion
```

Forward chaining dispara R1, R2, R3 en ese orden (cada conclusión habilita la
siguiente) y alcanza el punto fijo con 5 hechos. El campo `rules_fired` del
JSON es el **componente de explicación**: cada conclusión conserva la regla
que la produjo — la diferencia entre un sistema experto y un oráculo.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("logic", seed=22)
show(result)


## Reflexión

1. El laboratorio responde '¿qué se deriva de estos datos?' (forward). Formula la consulta inversa ('¿se requiere evaluación?') y describe la traza de backward chaining: ¿qué subobjetivos genera y en qué orden?
2. Si la base tuviera 10 000 reglas y llegara un hecho nuevo por segundo, ¿por qué el MATCH ingenuo colapsa y qué hace exactamente Rete para que el costo dependa del cambio y no del total?
3. Dos reglas independientes concluyen lo mismo con CF 0,6 y 0,5. La combinación da 0,8. ¿Qué supuesto oculto hay en ese número y qué pasaría si ambas reglas se basaran en el mismo síntoma?
